# sum-back-expand-broadcast — ex1: sum_back — unsqueeze + expand back to x.shape

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `sum-back-expand-broadcast`. Running the final beacon cell reports progress against the `Backprop: sum_back via expand_broadcast` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: sum_back via expand_broadcast` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sum-back-expand-broadcast`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sum-back-expand-broadcast"
DD_SUBTOPIC = "Backprop: sum_back via expand_broadcast"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `sum_back` via expand/broadcast — quick refresher

`sum(x, dim=k, keepdim=False)` collapses axis `k`. Each input entry `x[..., i, ...]` contributes to exactly ONE output entry, so the backward pass broadcasts `grad_out` back along the summed axis.

**Worked exemplar.**
```
x.shape         = (3, 4)
out = x.sum(dim=0)             # out.shape = (4,)
grad_out.shape  = (4,)
# 1. restore the collapsed axis as size-1:
g = grad_out.unsqueeze(0)      # (1, 4)
# 2. expand back to x's shape:
grad_in = g.expand(3, 4)       # broadcast — no copy
```

With `keepdim=True`, the unsqueeze step is skipped — `grad_out` already has the size-1 axis.

### Exercise 1 — sum_back — unsqueeze + expand back to x.shape

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the expand-broadcast pattern: restore the collapsed axis (if keepdim=False) then broadcast grad_out back to x.shape.
> Keywords: sum, expand, broadcast, keepdim
> ```

**KCs targeted:** `sum-backward-pattern`, `kwargs-pass-through-recipe`

Implement `sum_back(grad_out, out, x, dim, keepdim=False)` for the forward op `out = x.sum(dim=dim, keepdim=keepdim)`.

Derivation:
- Each input entry `x[..., i, ...]` contributes to ONE output entry. Local derivative is 1 at every position.
- Backward broadcasts `grad_out` back along the summed axis.

Two cases:

**`keepdim=True`** — `grad_out` already has a size-1 axis at `dim`. Just `expand` to `x.shape`:
```
grad_in = grad_out.expand(x.shape)
```

**`keepdim=False`** — `grad_out` is missing the axis. Unsqueeze it back to size 1, then expand:
```
g = grad_out.unsqueeze(dim)
grad_in = g.expand(x.shape)
```

Return a `torch.Tensor` with the same shape as `x`. `dim` is a single int (no multi-dim sum in this drill). No autograd.

In [ ]:
def sum_back(grad_out: Tensor, out: Tensor, x: Tensor, dim: int, keepdim: bool = False) -> Tensor:
    """dL/dx for out = x.sum(dim=dim, keepdim=keepdim)."""
    raise NotImplementedError()


def _test_ex1():
    # --- sum dim=0, keepdim=False ---
    x = t.arange(12.0).reshape(3, 4)
    out = x.sum(dim=0)                  # (4,)
    grad_out = t.tensor([1.0, 2.0, 3.0, 4.0])
    g = sum_back(grad_out, out, x, dim=0, keepdim=False)
    assert g.shape == (3, 4), f'shape: {g.shape}'
    # Every row of g is grad_out broadcast.
    assert t.allclose(g, grad_out.unsqueeze(0).expand(3, 4))

    # --- sum dim=1, keepdim=False ---
    out = x.sum(dim=1)                  # (3,)
    grad_out = t.tensor([10.0, 20.0, 30.0])
    g = sum_back(grad_out, out, x, dim=1, keepdim=False)
    assert g.shape == (3, 4)
    assert t.allclose(g, grad_out.unsqueeze(1).expand(3, 4))
    # Spot-check: row 0 of g must be [10, 10, 10, 10].
    assert t.allclose(g[0], t.full((4,), 10.0))

    # --- sum dim=0, keepdim=True ---
    out_kd = x.sum(dim=0, keepdim=True)  # (1, 4)
    grad_out_kd = t.tensor([[1.0, 2.0, 3.0, 4.0]])
    g = sum_back(grad_out_kd, out_kd, x, dim=0, keepdim=True)
    assert g.shape == (3, 4)
    assert t.allclose(g, grad_out_kd.expand(3, 4))

    # --- 3D ---
    rng = t.Generator().manual_seed(0)
    X = t.randn(2, 3, 4, generator=rng)
    for d in range(3):
        OUT = X.sum(dim=d)
        G = t.randn(*OUT.shape, generator=rng)
        g = sum_back(G, OUT, X, dim=d, keepdim=False)
        assert g.shape == X.shape
        assert t.allclose(g, G.unsqueeze(d).expand(*X.shape)), f'dim={d}'

    # --- witness vs torch.autograd ---
    x_ref = t.randn(3, 4, requires_grad=True, generator=t.Generator().manual_seed(5))
    y = x_ref.sum(dim=1).sum()
    y.backward()
    x_det = x_ref.detach()
    out_cached = x_det.sum(dim=1)
    g_ours = sum_back(t.ones(3), out_cached, x_det, dim=1, keepdim=False)
    assert t.allclose(g_ours, x_ref.grad, atol=1e-6), 'disagrees with autograd'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def sum_back(grad_out: Tensor, out: Tensor, x: Tensor, dim: int, keepdim: bool = False) -> Tensor:
    # Restore the collapsed axis if keepdim=False, then broadcast.
    g = grad_out if keepdim else grad_out.unsqueeze(dim)
    return g.expand(x.shape)
```

**Why `expand` not `repeat`.** `expand` returns a view — no memory allocation, just a stride trick. `repeat` copies. For the reverse pass, expand is the right call: downstream ops that read the broadcast grad don't need a contiguous buffer.

**Why kwargs matter here.** This is the exemplar of why `Recipe` must store `kwargs`. If the forward was `x.sum(dim=1)` and you forgot to thread `dim=1` into the recipe, this back fn doesn't know which axis to restore. The shape error would surface here, not at the forward.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()